# Database index performance

Plot the latest completed index benchmark artifact: mean latency versus recall@10. Preserve the original exclusion of “Flat (brute-force)” and retain “No Index (baseline).”
This notebook does not run benchmarks or build indexes.

As in the planner model-performance figure, the best index for each embedding model is highlighted: one that no other index beats on recall@10 without also being slower, or on latency without also losing recall. The baseline is the exact-search reference rather than a candidate, so it is never highlighted.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
from analyses.helpers.publication import (
    DEFAULT_PALETTE,
    export_figure,
    load_settings,
    publication_style,
)
from matplotlib.lines import Line2D

settings = load_settings(ROOT)
publication_style()

In [ ]:
import seaborn as sns
from analyses.helpers.publication import benchmark_data

benchmark = benchmark_data(ROOT)
print(f"Benchmark source: {benchmark.attrs['source_run']}")

In [ ]:
BASELINE = "No Index (baseline)"
MODEL_MARKERS = {"UNICOM": "s", "CLIP": "o"}


def pareto_leaders(frame):
    """Rows no other row matches on recall and latency while beating it on one."""
    leaders = []
    for label, row in frame.iterrows():
        no_worse = (frame["recall@10"] >= row["recall@10"]) & (frame["avg_ms"] <= row["avg_ms"])
        strictly_better = (frame["recall@10"] > row["recall@10"]) | (
            frame["avg_ms"] < row["avg_ms"]
        )
        if not (no_worse & strictly_better).any():
            leaders.append(label)
    return leaders


# One leader set per embedding model: an index is chosen for a model, so it is
# only compared with the other indexes on the same embeddings.
indexed = benchmark.loc[benchmark["index"] != BASELINE]
leaders = [label for _, group in indexed.groupby("model") for label in pareto_leaders(group)]
benchmark["best_index"] = benchmark.index.isin(leaders)
indexes = list(dict.fromkeys(benchmark["index"]))
index_palette = dict(
    zip(indexes, sns.color_palette(DEFAULT_PALETTE, n_colors=len(indexes)), strict=True)
)

fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
for _, rows in benchmark.loc[benchmark["best_index"]].groupby("model"):
    if len(rows) > 1:
        frontier = rows.sort_values("avg_ms")
        ax.plot(
            frontier["avg_ms"],
            frontier["recall@10"],
            color="#333333",
            linewidth=1.25,
            alpha=0.45,
            zorder=1,
        )
for _, row in benchmark.iterrows():
    best = row["best_index"]
    ax.scatter(
        row["avg_ms"],
        row["recall@10"],
        color=index_palette[row["index"]],
        marker=MODEL_MARKERS[row["model"]],
        edgecolor="black" if best else "white",
        linewidth=1.6 if best else 0.8,
        s=150 if best else 75,
        alpha=1 if best else 0.72,
        zorder=3 if best else 2,
    )
ax.set(xlabel="Mean latency (ms, log scale)", ylabel="Recall@10", ylim=(0, 1.04))
ax.set_xscale("log")


def legend_marker(label, *, color, marker="s", edge="white", width=0.8, size=8):
    return Line2D(
        [0],
        [0],
        marker=marker,
        color="none",
        markerfacecolor=color,
        markeredgecolor=edge,
        markeredgewidth=width,
        markersize=size,
        label=label,
    )


legend_handles = [
    *(legend_marker(index.replace("_", "-"), color=index_palette[index]) for index in indexes),
    *(
        legend_marker(model, color="#333333", marker=marker)
        for model, marker in MODEL_MARKERS.items()
        if model in set(benchmark["model"])
    ),
    legend_marker("Best index", color="white", marker="o", edge="black", width=1.6, size=10),
]
# Inside the axes, where the low-recall/high-latency corner stays empty, so the
# legend costs no figure width.
ax.legend(
    handles=legend_handles,
    title="Index/model",
    loc="lower right",
    frameon=False,
    fontsize=10,
    title_fontsize=11,
)
sns.despine(ax=ax)
export_figure(fig, settings, "indexing_benchmark", {"measurements": benchmark})
plt.show()
plt.close(fig)